![interpreto_banner](../../assets/img/interpreto_banner.png)

# Classification Concept-based N-gram Explanation Tutorial

This tutorial shows how to extract and interpret concepts for a classification task using n-gram words, in two settings: **globally** and **per class**.

- **Globally**: we decompose the activations of the **entire** validation set to obtain **k concepts in total**.
- **Per class**: we decompose the activations of data from **each class separately**, obtaining **k concepts per class**. 

Several decomposition methods are available (e.g. Semi-NMF, PCA, etc.).
For more details, please refer to the [**Interpreto documentation**](https://for-sight-ai.github.io/interpreto/).

### How it works

**Extracting concepts**: we use `CLS_TOKEN` granularity, meaning only the CLS token activation of each input is kept. The decomposition (fit) is then performed on these CLS token activations, producing a concept space of k dimensions.

**Interpreting concepts with n-grams**: to find which words or groups of words best represent each concept, we extract all unique n-grams (groups of up to n consecutive words) from the dataset and pass each n-gram individually to the model. We take its CLS token activation and project it into the concept space obtained during the fit. The n-grams whose projections activate a concept the most are selected to represent the concept.

Using n-grams (2, 3, or more words) rather than single words seems to give more meaningful results: a single word passed alone to the model often lacks context, producing a less informative CLS activation.


## 1. Global n-gram concept interpretation

Extract **k concepts from all activations**, then interpret them with n-grams.

1. [➗ **Split** your model in two parts](#split-global)
2. [🚦 Compute **CLS token activations** on the whole test set](#activations-global)
3. [🏋️‍♂️ **Decompose** the CLS activation space to obtain k concepts](#fit-global)
4. [🏷️ **Interpret** each concept by projecting n-gram activations into the concept space](#interpret-global)
5. [📊 Compute the **importance** of each concept for each class](#important-global)
6. [🔍 **Visualize** concept importance and n-gram labels](#visualize-global)

## 2. Class-wise n-gram concept interpretation

Extract **k concepts per class** by decomposing only the CLS activations of data from that class, then interpret them with n-grams.

1. [🚦 Compute **CLS token activations** for each class separately](#activations-perclass)
2. [🏋️‍♂️ **Decompose** the CLS activation space of each class to obtain k concepts per class](#fit-perclass)
3. [🏷️ **Interpret** each concept by projecting n-gram activations into the concept space](#interpret-perclass)
4. [📊 Compute the **importance** of each concept for its class](#important-perclass)
5. [🔍 **Visualize** concept importance and n-gram labels per class](#visualize-perclass)


In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# utils
import json

from IPython.display import HTML, display


def display_topngram_dropdown(topngram_words):
    data_json = json.dumps(
        {str(k): [{"ngram": ng, "score": f"{s:.4f}"} for ng, s in v.items()] for k, v in topngram_words.items()}
    )

    options_html = "".join(f'<option value="{k}">{k}</option>' for k in sorted(topngram_words.keys()))

    html = f"""
    <select id="concept-select" onchange="updateTable()">
    {options_html}
    </select>
    <table id="concept-table" border="1" style="margin-top:8px"></table>
    <script>
    var data = {data_json};
    function updateTable() {{
        var key = document.getElementById('concept-select').value;
        var rows = data[key];
        var html = '<tr><th>ngram</th><th>score</th></tr>';
        rows.forEach(function(r) {{ html += '<tr><td>'+r.ngram+'</td><td>'+r.score+'</td></tr>'; }});
        document.getElementById('concept-table').innerHTML = html;
    }}
    updateTable();
    </script>
    """
    display(HTML(html))

# 1. Global n-gram concept interpretation

## 1. ➗ **Split** your model in two parts <a class="anchor" id="split"></a>

We choose a `bert` fine-tuned on the `go-emotions` dataset and split it just before the classification head.

To split the model, we use the [`interpreto.ModelWithSplitPoints`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/) which wraps around the `transformers` model and allows the computation of activations at the specified `split_point`.

In [2]:
from transformers import AutoModelForSequenceClassification

from interpreto import ModelWithSplitPoints

model_with_split_points = ModelWithSplitPoints(
    model_or_repo_id="SchuylerH/bert-multilingual-go-emtions",
    automodel=AutoModelForSequenceClassification,
    split_point=5,  # split at the sixth layer
    device_map="cuda",
    batch_size=32,
)

## 2. 🚦 Compute a datasets of **activations** <a class="anchor" id="activations"></a>

We load the first 10000 documents of the `go_emotions` validation set.

Then we extract the activations of the [CLS] token of each document.

> ➡️ **Common practice**
>
> In the literature, to train concepts for classification it is common to use the [CLS] just before the classification head.
>
> In fact, at this layer, it makes no sense to use other elements.

> ⚠️ **Warning**
>
> In this notebook, many things are specific to the use of the [CLS] token.

[`interpreto.ModelWithSplitPoints.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/#interpreto.ModelWithSplitPoints.get_activations)

In [3]:
from datasets import load_dataset

# load the AG-News dataset
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
inputs = list(dataset["validation"]["text"])
classes_names = dataset["validation"].features["labels"].feature.names

# Compute the [CLS] token activations
granularity = ModelWithSplitPoints.activation_granularities.CLS_TOKEN  # required for ngram interpretation.
activations, predictions = model_with_split_points.get_activations(
    inputs=inputs,
    activation_granularity=granularity,
    include_predicted_classes=True,
)

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 3. 🏋️‍♂️ **Fit** a concept model on activations <a class="anchor" id="fit"></a>

With activations, we can train a concept model to find patterns (concepts).

The `concept_model` is an attribute of our concept explainer, similarly to the `model_with_split_points`. With these these two elements, we can go from inputs to concepts and from concepts to outputs.

In this tutorial, we use [`interpreto.concepts.SemiNMFConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/optim/#interpreto.concepts.SemiNMFConcepts). There are at least 15 others concept model available in interpreto. do not hesitate to explore them.

In [4]:
from interpreto.concepts import SemiNMFConcepts

# instantiate the concept explainer
concept_explainer = SemiNMFConcepts(
    model_with_split_points, nb_concepts=50, device="cuda"
)  # nb: nb_concept must be bigger than the number of classes, since there should be at least 1 concept = 1 class

# fit the concept explainer on activations
concept_explainer.fit(activations)

## 4. 🏷️ **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

We need to make sense of these concepts. In this case, we will use the [`interpreto.concepts.interpretations.TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/concepts_interpretations/#interpreto.concepts.interpretations.TopKInputs) to find the k (here, 5) ngram words (here 6 grams) which activates the most our concepts.

The `unique_words_kwargs` parameter controls how n-grams are extracted from the dataset before being passed to the model:

- **`count_min_threshold`**: minimum number of occurrences for an n-gram to be kept. Increase this value if irrelevant or noisy n-grams appear in the results; decrease it if meaningful n-grams are filtered out.
- **`lemmatize`**: if `True`, words are reduced to their base form before forming n-grams (e.g. "running" → "run", "cats" → "cat"). This avoids duplicates like "run" and "running" being treated as separate n-grams.
- **`words_to_ignore`**: list of words to exclude before forming n-grams (e.g. punctuation, stopwords, or domain-specific noise).



In [5]:
from interpreto.concepts.interpretations import TopKInputs

# instantiate the interpretation method with the concept explainer
topngram_inputs_method = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    activation_granularity=granularity,
    use_unique_words=6,  # 6-grams words. Put 1 to get unique words only. With the [CLS] token granularity, we are forced to use unique words
    unique_words_kwargs={
        "count_min_threshold": round(
            len(inputs) * 0.002
        ),  # appear in at least 0.2% of the samples | increase if random ngrams appear and decrease if some ngram appear too often
        "lemmatize": True,
        "words_to_ignore": [],  # include noise words and punctuation
    },
)

In [6]:
topngram_words = topngram_inputs_method.interpret(
    inputs=inputs,
    concepts_indices="all",
)
display_topngram_dropdown(topngram_words)

In [7]:
import torch

# estimate the importance of concepts for each class using the gradient
gradients = concept_explainer.concept_output_gradient(
    inputs=inputs,
    targets=None,  # None means all classes
    activation_granularity=granularity,
    concepts_x_gradients=True,  # the concept to output gradients are multiplied by the concepts values, this is common practice in the literature
    batch_size=32,
)

# stack gradients on samples and average them over samples
mean_gradients = torch.stack(gradients).abs().squeeze().mean(0)  # (num_classes, num_concepts)

# for each class, sort the importance scores
order = torch.argsort(mean_gradients, descending=True)

# visualize the top 5 concepts for each class
for target in range(order.shape[0]):
    print(f"\nClass: {classes_names[target]}:")
    for i in range(5):
        concept_id = order[target, i].item()
        importance = mean_gradients[target, concept_id].item()
        words = list(topngram_words.get(concept_id, None).keys())
        print(f"\tconcept id: {concept_id},\timportance: {round(importance, 3)},\ttopk words: {words}")


Class: admiration:
	concept id: 19,	importance: 0.037,	topk words: ['>', 'happen', 'why', 'out', 'i appreciate']
	concept id: 20,	importance: 0.034,	topk words: ['>', 'else', '/s', ': (', ')']
	concept id: 33,	importance: 0.033,	topk words: ['lol', 'lol .', 'glad', 'glad to', 'haha']
	concept id: 48,	importance: 0.028,	topk words: ['to your', 'for your', 'in your', 'to you', 'from the']
	concept id: 5,	importance: 0.027,	topk words: ['working', 'very', 'quite', 'care about', 'talking']

Class: amusement:
	concept id: 37,	importance: 0.027,	topk words: ["n't believe", '* *', "n't think", 'relationship', '*']
	concept id: 49,	importance: 0.026,	topk words: ['have been', 'made', 'it out', 'leave', 'become']
	concept id: 42,	importance: 0.026,	topk words: ['i love this', 'i love it', 'i love the', 'love it', 'love this']
	concept id: 20,	importance: 0.026,	topk words: ['>', 'else', '/s', ': (', ')']
	concept id: 15,	importance: 0.025,	topk words: ['lol', 'lol ,', 'haha', 'lol .', 'oh']

C

In [8]:
from interpreto import plot_concepts

labels = {k: list(v.keys()) for k, v in topngram_words.items()}

plot_concepts(
    classes_names=classes_names,
    concepts_importances=mean_gradients,
    concepts_labels=labels,
)

## 2. Class-wise n-gram concept interpretation

In the global approach above, we decompose the activations of the **entire** dataset, producing k concepts shared across all classes. Here, we take a different approach: for each class, we select only the data points predicted as that class, and decompose **their activations separately**.

This yields **k concepts per class**, each capturing patterns specific to that class. For example, in a sentiment analysis task, a "joy" class might have concepts related to celebration, gratitude, or humor, while a "sadness" class might have concepts related to loss, loneliness, or disappointment.

The steps are the same as before (decompose, interpret with n-grams, compute importance), but repeated independently for each class. The model split and CLS token activations computed earlier are reused — only the decomposition and interpretation are class-specific.


In [9]:
concept_explainers = {}
concept_interpretations = {}
concept_importances = {}

# iterate over classes
for target, class_name in enumerate(classes_names):
    # ----------------------------------------------------------------------------------------------
    # 2. construct the dataset of activations (extract the ones related to the class)
    indices = (predictions == target).nonzero(as_tuple=True)[0]
    class_wise_inputs = [inputs[i] for i in indices]
    class_wise_activations = activations[indices]

    if len(indices) == 0:
        print(f"\nClass: {class_name} — no samples predicted, skipping.")
        continue

    # ----------------------------------------------------------------------------------------------
    # 3. train concept model
    concept_explainers[target] = SemiNMFConcepts(model_with_split_points, nb_concepts=20, device="cuda")
    concept_explainers[target].fit(class_wise_activations)

    # ----------------------------------------------------------------------------------------------
    # 4. get the k-ngram-words activating the most the concept
    topngram_inputs_method = TopKInputs(
        concept_explainer=concept_explainers[target],
        k=15,
        activation_granularity=granularity,
        use_unique_words=5,  # 5-grams words to illustrate each concept
        unique_words_kwargs={
            "count_min_threshold": round(len(inputs) * 0.002),
            "lemmatize": True,
            "words_to_ignore": [],
        },
    )
    concept_interpretations[target] = topngram_inputs_method.interpret(
        inputs=inputs,
        concepts_indices="all",
    )

    # ----------------------------------------------------------------------------------------------
    # 5. compute concepts importance
    gradients = concept_explainers[target].concept_output_gradient(
        inputs=class_wise_inputs,
        targets=[target],
        activation_granularity=granularity,
        concepts_x_gradients=True,
        batch_size=8,
    )

    concept_importances[target] = torch.stack(gradients, axis=0).squeeze().abs().mean(dim=0)

    important_concept_indices = torch.argsort(concept_importances[target], descending=True).tolist()

    print(f"\nClass: {class_name}")
    for concept_id in important_concept_indices[:5]:
        ngrams = concept_interpretations[target].get(concept_id)
        importance = concept_importances[target][concept_id].item()
        words = list(ngrams.keys()) if ngrams is not None else []
        print(f"\timportance: {round(importance, 3)},\tconcept {concept_id},\ttopk ngrams: {words}")


Class: admiration
	importance: 0.089,	concept 4,	topk ngrams: ['i can ’ t', "i 'm not", 'i don ’ t', "'re not", 'it ’ s not', 'i ’ m not', "it 's not", "i did n't", "'s not", 'favorite', "n't have", 'good luck', 'you can', "i do n't", 'cute']
	importance: 0.086,	concept 9,	topk ngrams: ['i thought it', 'i used', 'wrong .', '’ t think', "i did n't", 'i wish', 'i thought', 'i hate', 'i don ’ t', 'not .', 'it ’ s not', 'i can ’ t', 'wish', 'lol', 'why']
	importance: 0.082,	concept 6,	topk ngrams: ['than the', 'that [', 'if you', 'now', '] and [', 'of those', 'now i', 'about [', '] would', "if you 're", 'if we', 'to [', 'of them', 'out of', 'edit :']
	importance: 0.082,	concept 18,	topk ngrams: ['cute', 'great', 'good luck', 'awesome', 'pretty', 'awesome .', 'good .', 'good', 'nothing', 'is awesome', 'the best', 'cut', 'amazing', 'beautiful', 'huge']
	importance: 0.068,	concept 0,	topk ngrams: ["what 's", 'my favorite', 'favorite', '. what', 'is what', 'favourite', '’ s what', 'didn ’ t',

In [10]:
plot_concepts(
    classes_names=classes_names,
    concepts_importances={classes_names[t]: concept_importances[t] for t in concept_importances},
    concepts_labels={
        classes_names[t]: {
            cid: list(ngrams.keys()) for cid, ngrams in concept_interpretations[t].items() if ngrams is not None
        }
        for t in concept_interpretations
    },
    top_k=5,
)